2025.05

这是完整的流程，从h5ad文件开始，到组合扰动结束

包含了：
1. Trimming下载的h5ad文件，制作huggingface格式的dataset
2. 微调（细胞分类/扰动）
3. 细胞分类
3. 使用超参数优化完的模型，进行组合扰动
4. 扰动后数据处理

## `h5ad`Preprocessing

对数据进行预处理。
1. 预处理：将数据读入为anndata，设置var属性和obs属性：ENSEMBL gene ID为`ensembl_id`, read_counts为`n_counts`，加入一列稍后用于连接的id`joinid`
2. tokenizing：对预处理后的数据进行tokenizing（主要包括normalizing-rank）转为模型输入

In [ ]:
import json
import os
import pandas as pd
import scanpy as sc

#import cellxgene_census
import datasets
import numpy as np

ENSEMBL biomart，找gene和ENSEMBL的对应关系

In [ ]:
# 代理的两种方法
import requests
import os

# 方法1.1：在代码中禁用代理
session = requests.Session()
session.trust_env = False  # 不读取系统代理配置
response = session.get("https://example.com")
# 方法1.2：清除系统环境变量
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)
os.environ.pop('all_proxy', None)

# 使用pybiomart获取Ensembl基因ID和基因名的映射
from pybiomart import Dataset

# 连接ENSEMBL biomart
dataset1 = Dataset(name='hsapiens_gene_ensembl',
                  host='http://www.ensembl.org')
# 连接到特定版本的Ensembl
dataset2 = Dataset(name='hsapiens_gene_ensembl',
                 host='http://grch37.ensembl.org')  # 旧版本
# 获取映射表
mapping1 = dataset1.query(attributes=['external_gene_name', 'external_synonym', 'ensembl_gene_id'])
mapping2 = dataset2.query(attributes=['external_gene_name', 'external_synonym', 'ensembl_gene_id'])
# 转换为字典
dict1 = dict(zip(mapping1['Gene name'], mapping1['Gene stable ID']))
dict2 = dict(zip(mapping2['Gene name'], mapping2['Gene stable ID']))
# 合并两个字典,取并集
gene_to_ensembl = dict1 | dict2
#gene_to_ensembl
print(f"length of gene_to_ensembl: {len(gene_to_ensembl)} \n length of dict1: {len(dict1)} \n length of dict2: {len(dict2)}")

In [ ]:
# 读取h5ad文件
data_dir = "~/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/"
raw_data = data_dir + "rawdata/"

adataCD = sc.read_h5ad(raw_data + "Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time.h5ad")
adataCD

In [ ]:
# geneformer要求的数据集features
adata.var["ensembl_id"] = adataCD.var.index
adata.obs["n_counts"] = adataCD.X.sum(axis=1)
adata.obs["joinid"] = list(range(adataCD.n_obs))

#valid_cell_types = ["colonocyte", "colon epithelial cell", "intestine goblet cell", "epithelial cell"]
#adataCD_cell_types = adataCD[adataCD.obs['cell_type'].isin(valid_cell_types)].copy() # 保留valid_cell_types里的细胞
#adataCD_cell_types.obs['Age_group'].value_counts(dropna = False)
#valid_Age_group = ["Pediatric", "Pediatric_IBD"]
#adataCD = adataCD[adataCD.obs['Age_group'].isin(valid_Age_group)].copy() # 只保留成人的细胞 
# 这里发现.obs['self_reported_ethnicity'].value_counts(dropna = False) 只剩下unknown，没有European，所以没有种族筛选

In [ ]:
adataCD

In [ ]:
# 预处理结果保存为h5ad格式
h5ad_dir = data_dir + "h5ad/"
if not os.path.exists(h5ad_dir):
    os.makedirs(h5ad_dir)
adataCD.write(h5ad_dir + "Total_Cells_of_the_human_intestinal_tract_fullcells.filtered.h5ad")

In [ ]:
# 用于制作数据集中检查测试的命令
adataCD_cell_types.obs['self_reported_ethnicity'].value_counts(dropna = False)

In [ ]:
%reset -f

1.	Adult (成人): 29146 个样本：这是最多的年龄组，可能代表成年人的数据集，样本数量较大。

2.	Second trim (妊娠第二三个月): 13782 个样本：这个组表示怀孕第二三个月的女性样本，样本数量相对较少。

3.	First trim (妊娠头三个月): 11221 个样本：这个组表示怀孕头三个月的女性样本，样本数量比第二三个月的组要少。

4.	`Pediatric` (儿童): 6842 个样本：该组表示儿童的样本，数量较成人组少，但仍然是一个相对较大的组。

5.	`Pediatric_IBD` (儿童炎症性肠病): 3504 个样本：该组是患有炎症性肠病（IBD）的儿童样本，数量相对较少。

6.	Adult_MLN (成人与MLN相关): 5 个样本：该组是与成人多发性淋巴结（MLN）相关的样本，样本数非常少。
    
7.	Second trim_MLN (妊娠第二三个月与MLN相关): 3 个样本：该组是与妊娠第二三个月相关的多发性淋巴结（MLN）样本，样本数非常少。

## Tokenization

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import json
import os
import pandas as pd

#import cellxgene_census
import datasets
import numpy as np
import scanpy as sc
import pickle
from torch.optim import AdamW

from geneformer import (
    DataCollatorForCellClassification,
    EmbExtractor,
    TranscriptomeTokenizer,
)
from transformers import BertForSequenceClassification, Trainer
import matplotlib.pyplot as plt

数据在 `~/geneformer_testdata/`

```
~/geneformer_testdata 
├── colon_epithelial                                                              # 数据集
│   ├── h5ad                                                                      # h5ad文件
│   └── tokenized.dataset                                                         # Tokenization后的dataset
├── colon_epithelial.geneformer.h5ad
├── ibd_iRiGiSresult_withGTEx_FinalFiltered.EAS80.tsv
├── in_silico_perturber.py
├── isp                                                                           # isp扰动文件夹
│   ├── isp_intermediate_files
│   ├── ispstats
│   ├── output_embs
│   └── outputs_intermediate_files
├── __pycache__
│   └── in_silico_perturber.cpython-310.pyc
└── Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time
    ├── 30m_tokenized.dataset
    ├── archive
    ├── fullcells_30m_tokenized.dataset
    ├── h5ad
    └── rawdata
```

In [ ]:
inputpath = "~/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/" # 数据集文件夹
h5ad_dir = inputpath + "h5ad/" # h5ad存放文件夹
token_dir = inputpath # Tokenization后的文件夹

In [ ]:
# 实例化tokenizer
# 这里使用geneformer_gc30M系列的模型，tokenizing也要相应修改
geneformer_gc30M_path = '~/Geneformer/geneformer/gene_dictionaries_30m/'
#geneformer_gc30M_path = '/data/home/user2/Geneformer/geneformer/gene_dictionaries_30m/'文件不全

tokenizer = TranscriptomeTokenizer(custom_attr_name_dict={"joinid": "joinid", "disease":"disease", "sex":"sex", "tissue":"tissue", "development_stage":"development_stage","cell_type":"cell_type"},
                                   model_input_size = 2048,
                                   special_token = False,
                                   nproc=40,
                                   gene_median_file = geneformer_gc30M_path + 'gene_median_dictionary_gc30M.pkl', # 用于数据预处理阶段，帮助标准化基因表达数据
                                   token_dictionary_file = geneformer_gc30M_path + 'token_dictionary_gc30M.pkl', # 用于分词阶段，将基因名称转换为模型能够理解的 token
                                   gene_mapping_file = geneformer_gc30M_path + 'ensembl_mapping_dict_gc30M.pkl') # 用于数据准备阶段，确保基因标识符的一致性
# 对数据进行tokenizing，并保存
tokenizer.tokenize_data(
    data_directory=h5ad_dir,
    output_directory=token_dir,
    output_prefix="fullcells_30m_tokenized",
    file_format="h5ad",
)

[老鼠模型](https://github.com/machine-perception-robotics-group/Mouse-Geneformer/)

In [ ]:
adata = sc.read_h5ad("~/geneformer_testdata/xiwj_data/h5ad/seurat_obj.singler.h5ad")
GeneSymbol_to_EnsemblID_dictionary = "~/mouse-geneformer/token_dict/MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID.pkl"
#token_dictionary = "~/Geneformer/geneformer/gene_dictionaries_30m/ensembl_mapping_dict_gc30M.pkl"
try:
    with open(GeneSymbol_to_EnsemblID_dictionary, 'rb') as file:
        GeneSymbol_to_EnsemblID = pickle.load(file)
except FileNotFoundError:
    print(f"文件 {token_dictionary} 不存在。")
except pickle.UnpicklingError as e:
    print(f"解包文件时出错: {e}")
adata

In [ ]:
adata.var['gene_name'] = adata.var['feature_name'].str.split('_').str[0]
adata.var["ensembl_id"] = adata.var["gene_name"].map(gene_to_ensembl)
adata.var.index = adata.var["ensembl_id"].values
adata = adata[:, ~adata.var['ensembl_id'].isna()].copy()
adata.var_names_make_unique()
adata.obs["n_counts"] = adata.X.sum(axis=1)
adata.obs["joinid"] = list(range(adata.n_obs))

In [ ]:
adata.write("~/geneformer_testdata/xiwj_data/h5ad/seurat_obj.singler.geneformer.h5ad")

In [ ]:
from scanpy import read_h5ad
adata = read_h5ad("~/geneformer_testdata/xiwj_data/h5ad/seurat_obj.singler.geneformer.h5ad")
adata

In [ ]:

inputpath = "~/geneformer_testdata/xiwj_data/" # 数据集文件夹
h5ad_dir = inputpath + "h5ad/" # h5ad存放文件夹
token_dir = inputpath # Tokenization后的文件夹
geneformer_gc30M_path = '~/mouse-geneformer/token_dict/'
tokenizer = TranscriptomeTokenizer(custom_attr_name_dict={"joinid":"joinid", "SingleR_labels":"SingleR_labels"},
                                   model_input_size = 2048,
                                   special_token = False,
                                   nproc=40,
                                   gene_median_file = geneformer_gc30M_path + 'mouse_gene_median_dictionary.pkl', # 用于数据预处理阶段，帮助标准化基因表达数据
                                   token_dictionary_file = geneformer_gc30M_path + 'MLM-re_token_dictionary_v1.pkl', # 用于分词阶段，将基因名称转换为模型能够理解的 token
                                   gene_mapping_file = geneformer_gc30M_path + 'MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID_inverted.pkl') # 用于数据准备阶段，确保基因标识符的一致性

tokenizer.tokenize_data(
    data_directory=h5ad_dir,
    output_directory=token_dir,
    output_prefix="xiwj_30m_tokenized",
    file_format="h5ad",
)

In [ ]:
%reset -f

## 细胞分类

In [ ]:
# imports
import datasets
from datasets import load_from_disk
from transformers import BertConfig, BertForSequenceClassification
from geneformer import model_info
import pickle
import os
import json
from transformers import Trainer
from geneformer import DataCollatorForCellClassification
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
from collections import Counter
import datetime

import subprocess
import seaborn as sns; sns.set()
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from transformers.training_args import TrainingArguments


import sys
import re
import numpy as np


import scanpy as sc

In [ ]:
# 被预测的数据集

dataset = datasets.load_from_disk("~/Projects/geneformer/Geneformer/Genecorpus-30M/genecorpus_30M_2048.dataset")
dataset = dataset.add_column("label", [0] * len(dataset))
dataset

In [ ]:
lengths = filtered_dataset["predicted_label"]
print("总样本数：", len(lengths))
#print("最小长度：", np.min(lengths))
#print("最大长度：", np.max(lengths))
#print("平均长度：", np.mean(lengths))
#print("中位数长度：", np.median(lengths))
#################################################################
#################################################################
"""
可视化 length 分布
"""
import matplotlib.pyplot as plt
plt.hist(lengths, bins=50, color='skyblue', edgecolor='black')
plt.title("Token Sequence Length Distribution")
plt.xlabel("Sequence Length")
plt.ylabel("Count")
plt.show()

In [ ]:
import pandas as pd
# Create a new dataset containing only records where length > 2047
from datasets import Dataset
from tqdm.auto import tqdm
from datasets import disable_progress_bar, enable_progress_bar

# 启用进度条
enable_progress_bar()

filtered_dataset = dataset.filter(
    lambda x: [length > 2047 for length in x['length']],
    with_indices=False,
    batched=True,
    num_proc=60,
    desc="Filtering dataset"
)

# 如果需要禁用进度条，可以使用：
# disable_progress_bar()

print(f"Original dataset size: {len(dataset)}")
print(f"Filtered dataset size: {len(filtered_dataset)}")
print(f"Percentage of records with length > 2047: {len(filtered_dataset)/len(dataset)*100:.2f}%")

在微调完的模型里选择适当的模型进行下游任务

In [ ]:
##模型准备
model_dir = "~/Geneformer/fine_tuned_models/fine_tuned_geneformer231215/"
model = BertForSequenceClassification.from_pretrained(model_dir)

model_info.print_model_config_info(model)
model_info.print_model_structure_summary(model, show_child_details=True)

geneformer_gc30M_path = '~/Geneformer/geneformer/gene_dictionaries_30m/'
with open(geneformer_gc30M_path + 'token_dictionary_gc30M.pkl', 'rb') as fp:
    token_dictionary = pickle.load(fp)

label_mapping_dict_file = os.path.join(model_dir, "label_to_cell_subclass.json")
with open(label_mapping_dict_file) as fp:
    label_mapping_dict = json.load(fp)
del label_mapping_dict_file

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    bf16=True,
    auto_find_batch_size=True,
    #per_device_train_batch_size=32,
    #per_device_eval_batch_size=32,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForCellClassification(token_dictionary=token_dictionary),
)
predictions = trainer.predict(filtered_dataset)
predictions.predictions.shape
# 处理预测结果
predicted_label_ids = np.argmax(predictions.predictions, axis=1)
predicted_logits = [predictions.predictions[i][predicted_label_ids[i]] for i in range(len(predicted_label_ids))]
predicted_labels = [label_mapping_dict[str(i)] for i in predicted_label_ids]

# filtered_dataset = filtered_dataset.add_column('predicted_label', predicted_labels)
# predicted_probs = np.exp(predicted_logits) / (1 + np.exp(predicted_logits))
# filtered_dataset = filtered_dataset.add_column('predicted_probs', predicted_probs)
# filtered_dataset.save_to_disk("/path/to/save/dataset_with_predictions")
# filtered_dataset.save_to_disk("~/geneformer_testdata/genecorpus_30M_2048_88predictions")

In [ ]:
filtered_dataset.save_to_disk("~/geneformer_testdata/genecorpus_30M_2048_88predictions")

In [ ]:
import scanpy as sc
# 定义输入的 h5ad 文件路径
inputh5adpath = "~/geneformer_testdata/xiwj_data/h5ad/seurat_obj.singler.geneformer.h5ad"

# 使用 scanpy 读取 h5ad 文件，加载为 AnnData 对象
adata = sc.read_h5ad(inputh5adpath)

# 将预测的细胞子类标签添加到 AnnData 的 obs 属性中
# "mouse_geneformer_predicted_cell_subclass" 是预测的细胞子类标签
adata.obs["mouse_geneformer_predicted_cell_subclass"] = predicted_labels

# 将预测的细胞子类概率添加到 AnnData 的 obs 属性中
# 使用 sigmoid 函数计算概率：exp(logits) / (1 + exp(logits))
adata.obs["mouse_geneformer_pedicted_cell_subclass_probability"] = np.exp(predicted_logits) / (1 + np.exp(predicted_logits))

# 将更新后的 AnnData 对象保存为新的 h5ad 文件
adata.write("~/geneformer_testdata/xiwj_data/feature.count.pro.30m.singler_withMouseGeneformer.h5ad")

In [ ]:
import scanpy as sc
inputh5adpath = "~/geneformer_testdata/xiwj_data/feature.count.pro.30m.singler_withMouseGeneformer.h5ad"
# 使用 scanpy 读取 h5ad 文件，加载为 AnnData 对象
adata = sc.read_h5ad(inputh5adpath)
del inputh5adpath

In [ ]:
##结果可视化
# 对单细胞数据进行预处理、降维
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

adata1 = adata[:, adata.var.highly_variable]
sc.pp.scale(adata1, max_value=10)
sc.tl.pca(adata1, svd_solver="arpack")

# 设置随机种子 (任意整数)
random_seed = 42
sc.pp.neighbors(adata1, n_neighbors=20, n_pcs=30, random_state=random_seed)
sc.tl.umap(adata1, random_state=random_seed)


In [ ]:
# 对 AnnData 对象 `adata1` 执行 Leiden 聚类
# `flavor='igraph'` 指定使用的算法实现
# `n_iterations=2` 设置聚类算法的迭代次数
sc.tl.leiden(adata1, flavor='igraph', n_iterations=2)

# 打印 `adata1` 的 `obs` 属性中 'leiden' 列的类别
# 这将显示分配给细胞的唯一聚类标签
print(adata1.obs['leiden'].cat.categories)

In [ ]:
# 查询细胞信息的函数
def query_cell_info(unique_labels):
    import obonet

    obo_file_path = '~/database_workshop/cl.obo'
    graph = obonet.read_obo(obo_file_path)
    cell_info = {}
    for term_id in unique_labels:
        label = graph.nodes[term_id].get('name')
        cell_id = str(term_id)
        cell_name = str(label)
        cell_info[cell_id] = cell_name
    return cell_info

def replace_labels_with_cell_info(adata, label_column:str):
    # 步骤 1: 提取指定 .obs 列的值并去重
    labels_unique = list(set(adata.obs[label_column]))

    labels_unique = [label.replace('_', ':') for label in labels_unique if label.startswith('CL')]
    # 步骤 2: 使用 cell_ontology_to_name 函数查询细胞信息
    cell_info = query_cell_info(labels_unique)

    # 步骤 3: 替换 .obs[label_column] 中的值
    adata.obs[label_column] = adata.obs[label_column].map(lambda label: cell_info.get(label, label))

    # 返回处理后的 adata
    return adata

# 假设 SingleR_labels 和 mouse_geneformer_predicted_cell_subclass 是相同的标签
adata1 = replace_labels_with_cell_info(adata1, "SingleR_labels")
adata1 = replace_labels_with_cell_info(adata1, "mouse_geneformer_predicted_cell_subclass")

In [ ]:
# 合并两个注释方法的所有唯一标签
all_labels = sorted(set(adata1.obs["SingleR_labels"].cat.categories).union(
                    set(adata1.obs["mouse_geneformer_predicted_cell_subclass"].cat.categories)))

# 创建标签到颜色的映射（确保所有标签都有唯一颜色）
color_map =  matplotlib.colormaps['tab20'].resampled(len(all_labels))
label_to_color = {label: color_map(i) for i, label in enumerate(all_labels)}

# 创建两个图的颜色列表
singleR_colors = [label_to_color[label] for label in adata1.obs["SingleR_labels"].cat.categories]
geneformer_colors = [label_to_color[label] for label in adata1.obs["mouse_geneformer_predicted_cell_subclass"].cat.categories]

# 创建颜色映射字典（这是关键！）
# 对于scanpy，我们需要创建一个标签到颜色的字典
singleR_palette = {label: label_to_color[label] for label in adata1.obs["SingleR_labels"].cat.categories}
geneformer_palette = {label: label_to_color[label] for label in adata1.obs["mouse_geneformer_predicted_cell_subclass"].cat.categories}

# 确保所有标签都有唯一颜色的映射
color_to_labels = {}  # 跟踪每个颜色被哪些标签使用

# 初始化颜色映射表
for palette in [singleR_palette, geneformer_palette]:
    for label, color in palette.items():
        if color not in color_to_labels:
            color_to_labels[color] = []
        color_to_labels[color].append(label)

# 检查并修正颜色冲突
for color, labels in list(color_to_labels.items()):
    if len(labels) > 1:  # 颜色被多个标签使用
        # 保留第一个标签的颜色，为其他标签分配新颜色
        for i, label in enumerate(labels[1:]):
            # 生成新颜色
            new_color_index = len(all_labels) + i  # 使用索引超出原始颜色映射范围
            new_color = plt.cm.get_cmap("tab20", len(all_labels) + len(labels))(new_color_index)
            
            # 更新调色板
            if label in singleR_palette:
                singleR_palette[label] = new_color
            if label in geneformer_palette:
                geneformer_palette[label] = new_color
            
            # 更新颜色映射表
            color_to_labels[color].remove(label)
            if new_color not in color_to_labels:
                color_to_labels[new_color] = []
            color_to_labels[new_color].append(label)

# 创建图形
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), dpi=600)

# 第一个图：SingleR_labels
sc.pl.umap(
    adata1,
    color="SingleR_labels",  # 使用字符串而不是列表
    title="SingleR_labels",
    ax=ax1,
    show=False,
    palette=singleR_palette  # 直接使用标签到颜色的字典
)

# 第二个图：Predicted Cell Subclass by Mouse Geneformer
sc.pl.umap(
    adata1,
    color="mouse_geneformer_predicted_cell_subclass",  # 使用字符串而不是列表
    title="Predicted Cell Subclass by Mouse Geneformer",
    ax=ax2,
    show=False,
    palette=geneformer_palette  # 直接使用标签到颜色的字典
)

# 调整布局
plt.tight_layout()
plt.show()
    

## 基因网络扰动

In [ ]:
import logging

import os
import pickle
from collections import defaultdict

import torch
from datasets import Dataset
from multiprocess import set_start_method
from tqdm.auto import trange
import datasets

#geneformer = "~/Geneformer/geneformer"
TOKEN_DICTIONARY_FILE = "token_dictionary_gc30M.pkl"
from geneformer import TOKEN_DICTIONARY_FILE
from geneformer import perturber_utils as pu
from geneformer.emb_extractor import get_embs
from geneformer.emb_extractor import EmbExtractor

from geneformer import InSilicoPerturber
from geneformer import InSilicoPerturberStats


datasets.logging.disable_progress_bar()
logger = logging.getLogger(__name__)

### 扰动计算`InSilicoPerturber`

分为三步
1. 对`.dataset`进行Embedding 

    `EmbExtractor.get_state_embs`

2. 预处理输入的基因对

    输出`[valid_combinations]: list`

3. 对每对基因对进行扰动计算

    `InSilicoPerturber.perturb_data`

---


In [ ]:
input_dataset = "~/Projects/geneformer/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/30m_tokenized.dataset"
embs_output_directory = "~/Projects/geneformer/geneformer_testdata/isp/output_embs"
token_dictionary = "~/Projects/geneformer/Geneformer/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl"

In [ ]:
# 设置扰动的细胞类型
#filter_data_dict={"cell_type":["colonocyte", "colon epithelial cell", "intestine goblet cell", "epithelial cell"]}
filter_data_dict={}

# 细胞状态的分类
cell_states_to_model={"state_key": "disease", 
                      "start_state": "normal", 
                      "goal_state": "Crohn disease", 
                      "alt_states": []}

# Embedding的设置
embex = EmbExtractor(model_type="CellClassifier", # if using previously fine-tuned cell classifier model
                     num_classes=2,
                     filter_data=filter_data_dict,
                     max_ncells=1000,
                     emb_layer=0,
                     summary_stat="exact_mean",
                     forward_batch_size=256,
                     nproc=20,
                     token_dictionary_file=token_dictionary,
                     emb_mode="cell")

# Embedding提取
state_embs_dict = embex.get_state_embs(cell_states_to_model=cell_states_to_model,
                                       model_directory="~/Projects/geneformer/finetuning30m/250516115628/250516_geneformer_cellClassifier_cm_classifier_test/ksplit1/_objective_2025-05-16_11-57-46/_objective_878622fc_12_learning_rate=0.0005,lr_scheduler_type=cosine,num_train_epochs=1,per_device_train_batch_size=12,seed=86.044_2025-05-16_13-27-01/checkpoint_000000/checkpoint-3761/",
                                       input_data_file=input_dataset,
                                       output_directory=embs_output_directory,
                                       output_prefix="control_to_CD")

筛选基因，选出在字典中的基因

In [ ]:
#genes_list2 = ["ENSG00000105329", "ENSG00000197919", "ENSG00000171855", "ENSG00000111537", "ENSG00000232810", "ENSG00000164400", "ENSG00000125538", "ENSG00000109471", "ENSG00000164399", "ENSG00000113520", "ENSG00000113525", "ENSG00000136244", "ENSG00000104432", "ENSG00000145839", "ENSG00000136634", "ENSG00000111537", "ENSG00000141800", "ENSG00000164914", "ENSG00000171862"]
# 常见IBD基因 TGF-β1, IFNα1, IFNβ1, IFNγ, TNF, GM-CSF, IL-1β, IL-2, IL-3, IL-4, IL-5, IL-6, IL-7, IL-9, IL-10, STAT1, STAT3, STAT4, JAK2
#genes_list = ['ENSG00000198626','ENSG00000155657','ENSG00000115641']
import pandas as pd
import numpy as np
genes_list = pd.read_csv("~/Documents/data/ibd/ibd_322_irigis_esgn.tsv",header=None, sep="\t").iloc[:,0].to_list()
#genes_list = ['ENSG00000145901','ENSG00000100330','ENSG00000113296','ENSG00000146094','ENSG00000095015','ENSG00000157593','ENSG00000253293','ENSG00000151150','ENSG00000116690','ENSG00000198561','ENSG00000152904','ENSG00000197959','ENSG00000152137','ENSG00000204842','ENSG00000140307','ENSG00000196177','ENSG00000230667','ENSG00000111424','ENSG00000188649','ENSG00000143437','ENSG00000151702','ENSG00000198561','ENSG00000166959','ENSG00000175567','ENSG00000162877','ENSG00000160791','ENSG00000164078','ENSG00000123066','ENSG00000253293','ENSG00000166473','ENSG00000130414','ENSG00000184588','ENSG00000143801','ENSG00000117620','ENSG00000107807','ENSG00000121380','ENSG00000183484','ENSG00000038532']

In [ ]:
import itertools
import pickle
# 读取token_dictionary
with open(token_dictionary, "rb") as f:
    gene_token_dict = pickle.load(f)
token_gene_dict = {v: k for k, v in gene_token_dict.items()}
pad_token_id = gene_token_dict.get("<pad>")
cls_token_id = gene_token_dict.get("<cls>")
eos_token_id = gene_token_dict.get("<eos>")

# 筛选出不在字典中的基因
def gene_filter_from_token_dict(genes_list, gene_token_dict):
    """
    Filters the input gene list based on the provided token dictionary.

    Args:
        genes_list (list): List of gene identifiers to filter.
        gene_token_dict (dict): Dictionary mapping gene identifiers to tokens.

    Returns:
        tuple: A tuple containing:
            - genes_list_filtered (list): Filtered list of genes present in the token dictionary.
            - missing_genes (list): List of genes not found in the token dictionary.
    """
    missing_genes = [
        gene
        for gene in genes_list
        if gene not in gene_token_dict.keys()
    ]
    if len(missing_genes) == len(genes_list):
        raise ValueError(
            "All genes are missing from the token dictionary. Please check the gene list or token dictionary."
        )
    elif len(missing_genes) > 0:
        print(f"Warning: The following genes are missing from the token dictionary: {missing_genes}")

    # Update the gene list to include only those present in the token dictionary
    genes_list_filtered = [gene for gene in genes_list if gene in gene_token_dict.keys()]
    print(f"Gene numbers to pertube: {len(genes_list_filtered)}/{len(genes_list)}")
    return genes_list_filtered, missing_genes

# Example usage
genes_list_filtered, missing_genes = gene_filter_from_token_dict(genes_list, gene_token_dict)

方案一：同细胞组合扰动

In [ ]:
# 方案一:同细胞组合扰动 —— 已迁至 geneformer_tools.gene_pairs.valid_pairs(已验证逻辑等价 17606==17606)
import datasets
from geneformer_tools.gene_pairs import valid_pairs
filtered_dataset = datasets.load_from_disk(input_dataset)
cell_sets = [set(ids) for ids in filtered_dataset["input_ids"]]
valid_combinations = [list(p) for p in valid_pairs(genes_list_filtered, gene_token_dict, cell_sets, ordered=False)]
print(f"Total valid gene combinations found: {len(valid_combinations)}")

方案二：异细胞组合扰动

In [ ]:
# 方案二:异细胞跨类型组合扰动 —— 已迁至 geneformer_tools.gene_pairs.valid_pairs_cross_celltype(已验证 25012==25012)
import datasets
from geneformer_tools.gene_pairs import valid_pairs_cross_celltype
filtered_dataset = datasets.load_from_disk(input_dataset)
isp_gene_cell_pair = {"cell1": "colonocyte", "cell2": "intestine goblet cell"}
valid_combinations = valid_pairs_cross_celltype(
    genes_list_filtered, gene_token_dict, filtered_dataset,
    isp_gene_cell_pair["cell1"], isp_gene_cell_pair["cell2"])
print(f"Total valid gene combinations found: {len(valid_combinations)}")

方案三：dual扰动


In [ ]:
# 方案三:dual 扰动(同细胞有序对)—— 已迁至 geneformer_tools.gene_pairs.valid_pairs(ordered=True,已验证 5628==5628)
import datasets
from geneformer_tools.gene_pairs import valid_pairs
filtered_dataset = datasets.load_from_disk(input_dataset)
cell_sets = [set(ids) for ids in filtered_dataset["input_ids"]]
valid_combinations = [list(p) for p in valid_pairs(genes_list_filtered, gene_token_dict, cell_sets, ordered=True)]
print(f"Total valid gene combinations found: {len(valid_combinations)}")

In [ ]:
# dual 扰动示例(单个基因对)—— 已迁至 geneformer_tools.isp_runner.run_isp_sweep
# 生产时对全部 valid_combinations 跑;这里仅示例第一个对。
from geneformer_tools import config
from geneformer_tools.isp_runner import run_isp_sweep
run_isp_sweep(
    gene_pairs=[valid_combinations[0]],
    perturb_type="dual",                      # -> DualPerturber.perturb_dual
    model_directory=config.FINETUNED_MODEL,
    input_data_file=input_dataset,
    output_directory="~/Projects/geneformer/geneformer_testdata/isp/tmpfile",
    state_embs_dict=state_embs_dict,
    token_dictionary_file=token_dictionary,
    cell_states_to_model=cell_states_to_model,
)

#### 基因对组合扰动计算

##### 同细胞组合扰动

In [ ]:
# overexpress sweep(遍历所有 valid_combinations)—— 已迁至 geneformer_tools.isp_runner.run_isp_sweep
from geneformer_tools import config
from geneformer_tools.isp_runner import run_isp_sweep, sweep_output_dir
_ox_out = sweep_output_dir(
    "~/Projects/geneformer/geneformer_testdata/isp/outputs_intermediate_files",
    "overexpress", cell_states_to_model)
run_isp_sweep(
    gene_pairs=valid_combinations,
    perturb_type="overexpress",               # -> InSilicoPerturber.perturb_data
    model_directory=config.FINETUNED_MODEL,
    input_data_file=input_dataset,
    output_directory=_ox_out,
    state_embs_dict=state_embs_dict,
    token_dictionary_file=token_dictionary,
    cell_states_to_model=cell_states_to_model,
)

#### `InSilicoPerturber`脚本原文

In [ ]:
# OF NOTE: token_dictionary_file must be set to the gc-30M token dictionary if using a 30M series model
# (otherwise the InSilicoPerturber will use the current default model dictionary)
# 30M token dictionary: https://huggingface.co/ctheodoris/Geneformer/blob/main/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl
# ['ENSG00000166825', 'ENSG00000169894']

isp = InSilicoPerturber(perturb_type="delete",
                        perturb_rank_shift=None,
                        genes_to_perturb=genes_list,
                        # 11868,12599,14262,24561
                        #genes_to_perturb=['ENSG00000178462','ENSG00000278535'],
                        #genes_to_perturb= "all",
                        combos=1,
                        anchor_gene=None,
                        model_type="CellClassifier", # if using previously fine-tuned cell classifier model
                        num_classes=2,
                        emb_mode="cell",
                        cell_emb_style="mean_pool",
                        filter_data=filter_data_dict,
                        cell_states_to_model=cell_states_to_model,
                        state_embs_dict=state_embs_dict,
                        token_dictionary_file=token_dictionary,
                        max_ncells=1000,
                        emb_layer=0,
                        forward_batch_size=10000,
                        nproc=1)
# outputs intermediate files from in silico perturbation
intermediate_files_fold = "~/Projects/geneformer/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/isp/outputs_intermediate_files"
isp.perturb_data(model_directory ="~/Projects/geneformer/finetuning30m/250516115628/250516_geneformer_cellClassifier_cm_classifier_test/ksplit1/_objective_2025-05-16_11-57-46/_objective_878622fc_12_learning_rate=0.0005,lr_scheduler_type=cosine,num_train_epochs=1,per_device_train_batch_size=12,seed=86.044_2025-05-16_13-27-01/checkpoint_000000/checkpoint-3761/", # example 30M fine-tuned model
                 #model_directory = "~/Geneformer/fine_tuned_models/gf-6L-30M-i2048_CellClassifier_cardiomyopathies_220224/",
                 input_data_file = input_dataset, # path/to/input_data
                 output_directory = intermediate_files_fold, # path/to/isp_output_directory
                 output_prefix = "control_to_CD_analysis")
# OF NOTE: token_dictionary_file must be set to the gc-30M token dictionary if using a 30M series model
# (otherwise the InSilicoPerturberStats will use the current default model dictionary)
# 30M token dictionary: https://huggingface.co/ctheodoris/Geneformer/blob/main/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl
ispstats = InSilicoPerturberStats(mode="goal_state_shift",
                                  genes_perturbed= genes_list,
                                  combos=1,
                                  anchor_gene=None,
                                  cell_states_to_model=cell_states_to_model,
                                  token_dictionary_file = token_dictionary)
ispstats.get_stats(input_data_directory = "~/Projects/geneformer/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/isp/outputs_intermediate_files", # this should be the directory 
                    null_dist_data_directory = None,
                    output_directory = "~/Projects/geneformer/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/isp/ispstats",
                    output_prefix = "ispstatas_finalresult")

### 扰动后数据统计`InSilicoPerturberStats`

从原脚本`geneformer/in_silico_perturber_stats.py`导入了`read_dictionaries`，`read_dict`，`get_gene_list`，`get_fdr`，`token_tuple_to_ensembl_ids`函数，更改了`isp_stats_to_goal_state`函数。

In [ ]:
# 扰动后统计 —— 用户改写的 isp_stats_to_goal_state 已迁至 geneformer_tools.isp_stats.compute_isp_stats
# (读 _raw.pickle 目录 -> 构造初始表 -> goal-state 偏移)。drop_gene_col=False 让下一格的 .iloc 沿用旧逻辑。
from geneformer_tools import config
from geneformer_tools.isp_stats import compute_isp_stats
genes_perturbed = "all"
cos_sims_df = compute_isp_stats(
    input_data_directory="~/Projects/geneformer/geneformer_testdata/isp/outputs_intermediate_files/2025_06_23_dual_[normal]-[Crohn disease]",
    cell_states_to_model=cell_states_to_model,
    genes_perturbed=genes_perturbed,
    token_dictionary_file=config.TOKEN_DICT_30M,
    gene_name_id_dictionary_file=config.GENE_NAME_ID_30M,
    drop_gene_col=False,
)
cos_sims_df

In [ ]:
# 保存扰动结果
from datetime import datetime
formatted_date = datetime.now().strftime("%y%m%d")

pertube_type = "dual" 
pertube_method = "combination"
dataset_name = "Total_Cells_of_the_human_intestinal"
cell_numbers = "10kcells"
pertube_direction = "N2CD"

isp_result_file_path = f"~/Projects/geneformer/geneformer_testdata/isp/ispstats/{pertube_direction}_{pertube_type}_{pertube_method}_{formatted_date}_[{dataset_name}]_{cell_numbers}.csv"

cos_sims_df = cos_sims_df.iloc[:, 1:]
cos_sims_df.to_csv(isp_result_file_path,sep = '\t',index=False)


### 扰动结果分析

1. 验证结果里的基因和OT的基因重合度
    1. 提取基因列表 P>0.05,N_Detections>500 
    2. 提取ibd里相关的前size=500个基因
    3. 过滤基因列表,获取评分
    4. 提取每个相关基因的文献证据

In [ ]:
from geneformer.mynewfun import ResultAnalysis
from itertools import combinations,chain
print("~/software/miniconda/envs/genefoemer/lib/python3.10/site-packages/geneformer/mynewfun.py")
print(ResultAnalysis.__doc__)
isp_stats_analysis = ResultAnalysis(isp_analysis_results_path = "~/geneformer_testdata/isp/ispstats/N2CD_dual_combination_250625_[Total_Cells_of_the_human_intestinal]_10kcells.csv",
                                    isp_gene_N_Detections_threshold=500)

isp_analysis_results_path = "~/geneformer_testdata/isp/ispstats/N2CD_delete_25-05-27_Total_Cells_of_the_human_intestinal_N2CD_10kcells_delete_combination_isp_result.csv"
isp_result_genepair_list,isp_result_gene_list = isp_stats_analysis.extract_isp_result_gene_lsit()
print(f"ips_result_pairs: {len(isp_result_genepair_list)}")

In [ ]:
log_buffer = []

log_buffer.append(f"REPORT\n\n"
                    f"Geneformer InSilicoPerturber gene pair number: {len(isp_result_genepair_list)}\n"
                    f"Gene number: {len(isp_result_gene_list)}"
                    )

In [ ]:
import os
import logging
filename = os.path.splitext(os.path.basename(isp_analysis_results_path))[0]
first_two_elements = filename.split('_')[:2]
new_filename = '_'.join(first_two_elements + ['isp_analysis'])
log_file_dir = "~/ipynb"
log_file_path = os.path.join(log_file_dir, f"{new_filename}.log")

logger = logging.getLogger('ResultAnalysisLogger')

In [ ]:
log_buffer = []
OT_ENSG2gene_dict, OT_score_dict, OT_ENSG2approvedSymbol_dict = isp_stats_analysis.query_disease_associated_targets(efo_id="EFO_0003767", size=1000)
isp_gene_list_OT_relative_ESGN, isp_gene_list_OT_relative_symbol = isp_stats_analysis.update_geneList_to_newestSymbolList(isp_result_gene_list,OT_ENSG2gene_dict,OT_ENSG2approvedSymbol_dict)
log_buffer.append(f"Within top 1k OpenTarget IBD related genes : {len(isp_gene_list_OT_relative_ESGN)}/{len(isp_result_gene_list)}")
annotated_gene_combinations = list(combinations(isp_gene_list_OT_relative_symbol, 2))
valid_combinations = [pair for pair in annotated_gene_combinations if tuple(sorted(pair)) in [tuple(sorted(p)) for p in isp_result_genepair_list]]
log_buffer.append(f"Valid gene pair number: {len(valid_combinations)}/{len(isp_result_genepair_list)}")
isp_valid_gene_list = list(set(chain.from_iterable(valid_combinations)))
isp_validgene_list_OT_relative_ESGN, isp_validgene_list_OT_relative_symbol = isp_stats_analysis.update_geneList_to_newestSymbolList(isp_valid_gene_list,OT_ENSG2gene_dict,OT_ENSG2approvedSymbol_dict)
log_buffer.append(f"valid gene number: {len(isp_validgene_list_OT_relative_ESGN)}/{len(isp_result_gene_list)}\n")
log_buffer

[作图](https://python-graph-gallery.com/web-ggbetweenstats-with-matplotlib/)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.font_manager")

BG_WHITE = "#faf1d9"
GREY_LIGHT = "#b4aea9"
GREY50 = "#7F7F7F"
BLUE_DARK = "#1B2838"
BLUE = "#2a475e"
BLACK = "#282724"
GREY_DARK = "#747473"
RED_DARK = "#850e00"
# Colors taken from Dark2 palette in RColorBrewer R library
COLOR_SCALE = ["#1B9E77", "#D95F02", "#7570B3"]

# 小提琴图的位置横坐标设置
POSITIONS = [1,1]

# 提取点的数据
y_data1 = list(OT_score_dict.values())
y_data2 = list(score for gene in isp_validgene_list_OT_relative_ESGN for score in [OT_score_dict.get(gene)] if score is not None)
y_data = [y_data1, y_data2]

jitter = 0.03
x_data = [np.array([1] * len(d)) for d in (y_data)]
np.random.seed(28) 
x_jittered = [np.clip(x + np.random.normal(0, jitter, len(x)), x - 0.1, x + 0.1) for x in x_data]

# Horizontal lines
HLINES = np.percentile(y_data1, [25, 50, 75]).tolist()

# 创建图例标签
legend_patches = [
    mpatches.Patch(color="#1B9E77", alpha=0.2, label="OT IDB assoc gene"),
    mpatches.Patch(color="#D95F02", alpha=0.8, label="isp gene"),
]

#---------------作图---------------

fig, ax = plt.subplots(figsize= (6, 8), dpi=600)
fig.patch.set_facecolor(BG_WHITE)
ax.set_facecolor(BG_WHITE)
ax.set_xticks([])
ax.set_xticklabels([])
ax.grid(False)
ax.set_title("Normal to CD Deletion isp Result", fontsize=14, color=BLUE_DARK)
# 添加图例
ax.legend(handles=legend_patches, loc="upper right", fontsize=10, frameon=False)


for h in HLINES:
    ax.axhline(h, color=GREY50, ls=(0, (3, 3)), alpha=0.3, zorder=0)

violins = ax.violinplot(
    y_data1, 
    positions=[1],
    widths=0.45,
    bw_method="silverman",
    showmeans=False, 
    showmedians=False,
    showextrema=False
)
for pc in violins["bodies"]:
    pc.set_facecolor("none")
    pc.set_edgecolor("#1B9E77")
    pc.set_linewidth(1)
    pc.set_alpha(0.2)
# Add jittered dots ----------------------------------------------
jitter_alpha = [0.2, 0.8]
linewidth_setting = [0.1,0.5]
for x, y, color, alpha,linewidth in zip(x_jittered, y_data, COLOR_SCALE, jitter_alpha, linewidth_setting):
    ax.scatter(x, y, s=25, color=color, alpha=alpha, edgecolor="black", linewidth=linewidth, zorder=3)
for gene_name, gene_y,gene_x in zip(isp_validgene_list_OT_relative_symbol, y_data2, x_jittered[1]):
    plt.text(gene_x + 0.009, gene_y, gene_name,
    va= 'center',
    fontsize=8, 
    color="#D95F02", 
    rotation=15, 
    rotation_mode='anchor')

plt.tight_layout()

In [ ]:
from geneformer.mynewfun import ResultAnalysis
#print("~/software/miniconda/envs/genefoemer/lib/python3.10/site-packages/geneformer/mynewfun.py")

isp_stats_analysis = ResultAnalysis(isp_analysis_results_path = "~/geneformer_testdata/isp/ispstats/CD2N_delete_25-05-29_Total_Cells_of_the_human_intestinal_CD2N_10kcells_delete_combination_isp_result.csv",
                                    isp_gene_N_Detections_threshold=500)
isp_stats_analysis.logging_literature_data()

## 检测&测试

#### 检测模型参数量

In [ ]:
from transformers import BertModel

model = BertModel.from_pretrained("~/finetuning30m/250516115628/250516_geneformer_cellClassifier_cm_classifier_test/ksplit1/_objective_2025-05-16_11-57-46/_objective_878622fc_12_learning_rate=0.0005,lr_scheduler_type=cosine,num_train_epochs=1,per_device_train_batch_size=12,seed=86.044_2025-05-16_13-27-01/checkpoint_000000/checkpoint-3761/")
total_params = sum(p.numel() for p in model.parameters())
print(f"BERT-base 参数量: {total_params:,}")  # 输出: 108,310,272 (108M)

#### 读取`.pickle`文件

In [ ]:
import pickle
import json

token_dictionary = "~/mouse-geneformer/token_dict/MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID.pkl"
#token_dictionary = "~/Geneformer/geneformer/gene_dictionaries_30m/ensembl_mapping_dict_gc30M.pkl"
try:
    with open(token_dictionary, 'rb') as file:
        data = pickle.load(file)
except FileNotFoundError:
    print(f"文件 {token_dictionary} 不存在。")
except pickle.UnpicklingError as e:
    print(f"解包文件时出错: {e}")
data

In [ ]:
token_dictionary = "~/mouse-geneformer/token_dict/MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID.pkl"
try:
    with open(token_dictionary, 'rb') as file:
        data = pickle.load(file)
except FileNotFoundError:
    print(f"文件 {token_dictionary} 不存在。")
except pickle.UnpicklingError as e:
    print(f"解包文件时出错: {e}")

# 读取完后，提取data的所有值，将其对应的键用值替换
if isinstance(data, dict):
    inverted_data = {v: v for k, v in data.items()}

# Save inverted_data to a new pickle file
output_path = token_dictionary.replace('.pkl', '_inverted.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(inverted_data, f)
print(f"Saved inverted dictionary to {output_path}")
inverted_data

#### 读取`.dataset`文件

In [ ]:
from datasets import load_from_disk

input_dataset = "~/Projects/geneformer/geneformer_testdata/Total_Cells_of_the_human_intestinal_tract_mapped_across_space_and_time/30m_tokenized.dataset"
try:
    dataset = load_from_disk(input_dataset)
    print(dataset)
except Exception as e:
    print(f"加载数据集时出现错误: {e}")

#### 查看`.bin`文件

In [ ]:
import torch

# 加载模型权重
state_dict = torch.load("~/mouse-geneformer/mouse-Geneformer/pytorch_model.bin", map_location="cpu")

# 打印出所有层的名称
for name in state_dict.keys():
    print(f"{name}: {state_dict[name].shape}")

for name, param in state_dict.items():
    if "classifier" in name and "weight" in name:
        print(f"{name}: {param.shape}")

#### 官方平台提供的模型接口

In [ ]:
from cellxgene_census.experimental import get_all_available_embeddings
CENSUS_VERSION = "2023-12-15"

for e in get_all_available_embeddings(CENSUS_VERSION):
    print(f"{e['embedding_name']:15} {e['experiment_name']:15} {e['data_type']:15}")

---